# Graph Neural Network Approach FDiGNN - Model Fitting and Node Importance

# Dataset: Malaria Dataset (MD)

LC-MS data obtained in Positive Ionization Mode. Samples are (almost all) triplicates of 273 human blood samples collected from 220 human subjects. A total of 840 samples were analysed.

- 237 samples infected with Malaria which had previously been infected without information on Chloroquine Resistance - used for analysis as **'P.Vivax+Prior'**
- 213 samples infected with Malaria which had not previously been infected without information on Chloroquine Resistance - used for analysis as **'P.Vivax+NoPrior'**
- 138 samples not infected with Malaria with no information on previous infections and on Chloroquine Resistance - 117 used for analysis as **'Control'**
- 99 samples infected with Malaria with no information on previous infections and susceptible to Chloroquine
- 45 samples not infected with Malaria which had not previously been infected without information on Chloroquine Resistance - used for analysis as **'Control'**
- 45 samples infected with Malaria which had previously been infected and were resistant to Chloroquine
- 45 samples infected with Malaria with no information on previous infections and resistant to Chloroquine
- 15 samples not infected with Malaria which had previously been infected without information on Chloroquine Resistance - used for analysis as **'Control'**
- 3 samples infected with Malaria which had not previously been infected and resistant to Chloroquine

### Notebook Organization:

- Reading Treated Data.
- Fitting RF and PLS-DA models (extracting feature importance and estimating model performance.
- Building Formula-Difference Networks and crete sample Formula-Difference Networks.
- Setting up the FDiGNN model.
- Fitting FDiGNN model (with all samples) and extract node/metabolite importance.
- Re-fitting RF and PLS-DA models with only peaks present in the FDiN.
- Comparing important metabolites to build each model.
- Highlighting Important Network Section and Pathway Enrichment Analysis.
- Dash App to Visualize Important Network Section and Overlap with different pathways.


- Getting Pathway graphlets and single nodes sets (with gaps) for the simulations*

## Due to stochasticity, re-running the notebook will get slightly different results. Thus, figures in the paper can be very slightly different.

### *- Important Note: Since the pathway+single nodes sets used for the paper were obtained with another run and even setting the random seed will not get equal graphlets to the ones used in the work, the default is to set this section to False so it does not affect posterior steps (it would be necessary to run new models instead of already using the previously trained models).

### Needed Imports

In [ ]:
# standard library imports
import pickle
import json
import sys
import time

# scientific python imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.model_selection import StratifiedKFold
from sklearn import model_selection
from sklearn.cross_decomposition import PLSRegression

import networkx as nx

# metabolinks and "in folder" modules
import metabolinks.transformations as transf
import metanalysis_standard as metsta
from multianalysis import _calculate_vips, _generate_y_PLSDA

import MDiN_functions as md

from tqdm import tqdm

import torch
from torch.nn import Linear, Softmax, Sequential, BatchNorm1d, ReLU, Dropout, LeakyReLU
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.utils.convert import from_networkx
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, TAGConv
from torch_geometric.nn import global_mean_pool, global_add_pool, global_max_pool

from torch_geometric.utils import softmax
from torch_scatter import scatter_add

# For Dash App
import networkx as nx
import dash_cytoscape as cyto
import dash
from dash import Dash, dcc, html, Input, Output, ctx, callback
import base64
from io import BytesIO

# Report versions
print("PyTorch version", torch.__version__)
print("CUDA version", torch.version.cuda)

torch.cuda.device_count()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'\nUSING DEVICE {device}\n')

# Reading Malaria Dataset Treated Data

In [ ]:
# Filename for the data to import
filename_TreatedData = 'Data/MD_AllTreatedData_Final.xlsx'
filename_proc = 'Data/MD_ProcData_Final.pickle'
filename_treat = 'Data/MD_TreatedData_Final.pickle'
target_name = 'Data/MD_Target_Final.txt'

bin_data = pd.read_excel(filename_TreatedData, sheet_name='BinSim Treated Data')#.set_index('Unnamed: 0').T
#univariate_data = pd.read_excel(filename_TreatedData, sheet_name='MVI+Norm Data')#.set_index('Unnamed: 0')
bin_data = bin_data.set_index('Bucket label').T
processed_data = pd.read_pickle(filename_proc)
treated_data = pd.read_pickle(filename_treat).T

bin_data.columns = [str(i) for i in treated_data.columns]

with open(target_name) as a:
    tg = a.readlines()
target = [t.strip() for t in tg]
sample_cols = list(treated_data.index)

# Set Up Train / Test Split and Details for Analysis

In [ ]:
iter_num = 10

np.random.seed(65824)
train_idxs, test_idxs, train_tg, test_tg = {}, {}, {}, {}
for a in range(1,1+iter_num):
    np.random.seed(65824*(a))
    train_idxs[a], test_idxs[a], train_tg[a], test_tg[a] = model_selection.train_test_split(
            treated_data.index, target, train_size=0.7, test_size=None, stratify=target,
    )

colours = sns.color_palette('tab10', 10) # Set the colors

# Random Forest

In [ ]:
# Choose a number for the seed for consistent results
np.random.seed(65824802)
n_trees=200 # Number of trees in the model

RF_accus = {}
#RF_imp_feats = {}

for a in train_idxs:
    RF_model = metsta.RF_model(treated_data.loc[train_idxs[a]],
                    train_tg[a], regres=False, # Data, labels and if it's a regression or classification
                    return_cv=False,
                    n_trees=n_trees, # Number of trees in the model
                    # Choose a method of cross-validation (None is stratified cv) and the number of folds
             metrics = ('accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted')) # Choose the performance metric

    rf_preds = RF_model.predict(treated_data.loc[test_idxs[a]])
    correct = 0
    for i in range(len(rf_preds)):
        if rf_preds[i] == test_tg[a][i]:
            correct += 1
    accuracy = correct/len(test_idxs[a])
    print(f'{a}:', accuracy)

    RF_accus[a] = accuracy
    #RF_imp_feats[a] = sorted(enumerate(RF_model.feature_importances_), key=lambda x: x[1], reverse=True)
    print(f'Finished fitting Random Forests for {a}.')

In [ ]:
print(f'RF Accuracy: {pd.Series(RF_accus).mean()} +- {pd.Series(RF_accus).std()}')

## PLS-DA

PLS-DA Component Number Optimization

In [ ]:
%%capture --no-stdout
# above is to supress PLS warnings

# Set the random seed
np.random.seed()

max_comp = 20 # Max. number of components to search (the higher the more time it takes)

# Store Results
PLS_optim = metsta.optim_PLSDA_n_components(treated_data, target, regres=False, # Data, target and if it's a regression
                                    encode2as1vector=True,
                                    max_comp=max_comp, # Max. number of components to search
                                    kf=None, n_fold=5, # Cross validation to use (none is stratified CV) and nº of folds
                                    scale=False) # Set scale to True only if you did not do scaling in pre-treatments

In [ ]:
scores_cols = sns.color_palette('tab10', 10) # Set the colors for the lines
with sns.axes_style("whitegrid"):
    with sns.plotting_context("notebook", font_scale=1.2):
        f, ax = plt.subplots(1, 1, figsize=(5,5), constrained_layout=True) # Set the figure size
        c = 0
        for i, values in PLS_optim.items():
            if i =='CVscores':
                name = 'Q$^2$'
            else:
                name = 'R$^2$'
            
            ax.plot(range(1, len(values) + 1), values, label=name, color = scores_cols[c])
            c = c+1
        
        ax.set(xlabel='Number of Components', # Set the label for the x axis
                ylabel='PLS Score') # Set the label for the Y axis
        ax.legend(loc='lower right', fontsize=15) # Set the legend
        ax.set_ylim([0, 1.02]) # Set limits for y axis
        ax.set_xticks(range(0, len(values), 2)) # Set ticks that appear in the bottom of x axis
        plt.show()

PLS-DA Model Fitting

In [ ]:
%%capture --no-stdout
# above is to supress PLS warnings
# Choose a number for the seed for consistent results
np.random.seed(65824802)

n_comp = 16 # Number of components of PLS-DA model - very important

PLSDA_accus = {}
#PLSDA_imp_feats = {}

for a in train_idxs:

    matrix_train = _generate_y_PLSDA(train_tg[a], pd.unique(target), False)
    matrix_test = _generate_y_PLSDA(test_tg[a], pd.unique(target), False)

    plsda = PLSRegression(n_components=n_comp, scale=False)
    # Fit PLS model
    plsda.fit(X=treated_data.loc[train_idxs[a]], Y=matrix_train)
    # Obtain results with the test group
    y_pred = plsda.predict(treated_data.loc[test_idxs[a]])

    accuracy = (matrix_test.idxmax(axis=1) == pd.DataFrame(
        y_pred, columns=matrix_test.columns).idxmax(axis=1)).sum()/len(matrix_test)

    PLSDA_accus[a] = accuracy
    #PLSDA_imp_feats[a] = sorted(enumerate(_calculate_vips(plsda)), key=lambda x: x[1], reverse=True)
    print(f'{a}:', accuracy)
    print(f'Finished fitting PLS-DA for {a}.')

In [ ]:
print(f'PLS-DA Accuracy: {pd.Series(PLSDA_accus).mean()} +- {pd.Series(PLSDA_accus).std()}')

# Graph Neural Network

## Prepare the Network

In [ ]:
# Get the list of formulas to build the sFDiN
temp_df = processed_data.copy()

for i in temp_df.index:
    # See formulas of annotated compounts
    fs = temp_df.loc[i, 'Matched HMDB formulas']
    if type(fs) == list:
        fs = list(set(fs))
        # If only one annotation, overwrite the formula with it
        if len(fs) == 1:
            temp_df.loc[i, 'Formula_Assignment'] = fs[0]
            # Should've been 'Matched HMDB adducts' here, did not catch that before
            temp_df.loc[i, 'Formula_Assignment Adduct'] = temp_df.loc[i, 'Matched HMDB formulas'][0]
        # If more than one annotation
        else:
            counted = False
            # If any of the annotated formulas is equal to the formula assigned, change it and keep it
            for f in fs:
                if f == temp_df.loc[i, 'Formula_Assignment']:
                    counted = True
            if counted == False:
                new_f = []
                # See formulas that only have C, H, O, N, S, P elements while having 1 C and 1 H at least
                for f in fs:
                    a = md.formula_process(f)
                    if a['C'] != 0 and a['H'] != 0:
                        if len(a) == 8:
                            if a['Cl'] == 0 and a['F'] == 0:
                                new_f.append(f)
                # If only 1 formula is in these conditions, overwrite formula assignment
                if len(new_f) == 1:
                    temp_df.loc[i, 'Formula_Assignment'] = new_f[0]


Put Formulas to build the FDiN in DataFrame format

In [ ]:
formula_df = temp_df
# Get the formulas from formula assignment, excluding isotopes
formula_df = formula_df.dropna(subset='Formula_Assignment')
formula_df = formula_df.loc[[i for i in formula_df.index if 'iso.' not in formula_df.loc[i, 'Formula_Assignment']]]
# Add the counts of the different elements in columns
elems = metsta.create_element_counts(formula_df, formula_subset=['Formula_Assignment',], compute_ratios=False,
                                     drop_duplicates=False)
filt_elems = elems.iloc[:,:-1]

Choose the list of Mass-Difference-based Building blocks (MDBs) to use and get them to DataFrame format

In [ ]:
# Create MDB list of accepted chemical transformations
MDB = ['H2','CH2','CO2','O','CH2O','NCH','O(N-H-)','S','CONH','PO3H','NH3(O-)','SO3','CO', 'C2H2O', 'H2O']
results = {}
for i in MDB:
    results[i] = md.formula_process(i, elems=filt_elems.columns)
MDB_df = pd.DataFrame(results).T

Read the built metabolic knowledge network and keep only the formula nodes that are present in the dataset

In [ ]:
with open('SMPDB_MetaNetwork_general.pickle', 'rb') as f:
    FDiN_knowledge = pickle.load(f)
node_list = list(FDiN_knowledge.nodes())

# See which of these formulas were detected in our dataset
keep_idxs = []
keep_formulas = []
keep_pathways = []
form_to_idx = {} # Dictionary so we know the association between formulas and idxs

for i in temp_df.index:
    counted = False
    # Give priority to annotated formulas
    fs = temp_df.loc[i, 'Matched HMDB formulas']
    if type(fs) == list:
        fs = list(set(fs))
        # Only 1 Formula assigned in annotated data
        if len(fs) == 1:
            form = fs[0]
            # If the formula is the knowledge network
            if form in node_list:
                keep_idxs.append(i)
                keep_formulas.append(form)
                if form in form_to_idx:
                    form_to_idx[form].append(i)
                else:
                    form_to_idx[form] = [i,]
                    keep_pathways.extend(FDiN_knowledge.nodes()[form]['Pathways'])
                counted = True
        # More than 1 Formula assigned in annotated data
        else:
            form_in_node_list = []
            for f in fs:
                if f in node_list:
                    form_in_node_list.append(f)
            if len(form_in_node_list) >= 1:
                for f in form_in_node_list:
                    keep_idxs.append(i)
                    keep_formulas.append(f)
                    if f in form_to_idx:
                        form_to_idx[f].append(i)
                    else:
                        form_to_idx[f] = [i,]
                        keep_pathways.extend(FDiN_knowledge.nodes()[f]['Pathways'])
                counted = True

    # Formula assignment Formulas
    if not counted:
        fs = temp_df.loc[i, 'Formula_Assignment']
        if type(fs) == str:
            if fs in node_list:
                keep_idxs.append(i)
                keep_formulas.append(fs)
                if fs in form_to_idx:
                    form_to_idx[fs].append(i)
                else:
                    form_to_idx[fs] = [i,]
                    keep_pathways.extend(FDiN_knowledge.nodes()[fs]['Pathways'])

# Subgraph the FDiN to only keep these formulas as information for FDiGNN
FDiN_knowledge = FDiN_knowledge.subgraph(keep_formulas)

## Building the FDiN proper

In [ ]:
# FDiN basis
FDiN = nx.Graph()
FDiN.add_nodes_from(filt_elems.index) # Each formula is a node
# Adding relevant attributes
nx.set_node_attributes(FDiN, formula_df['Formula_Assignment'].to_dict(), name='Formula')

# Adding simple edges
for formula in filt_elems.index:
    poss_formulas = filt_elems.loc[formula] + MDB_df
    for i in poss_formulas.index:
        poss_matches = filt_elems[(filt_elems == poss_formulas.loc[i]).sum(axis=1) == len(MDB_df.columns)]
        for node in poss_matches.index:
            FDiN.add_edge(formula, node, Transformation=i, Weight=1)

# Adding Knowledge-based edges from the metabolic-knowledge based network
for n1 in FDiN.nodes():
    formA = FDiN.nodes()[n1]['Formula']
    if formA in FDiN_knowledge.nodes():
        for n2 in FDiN.nodes():
            formB = FDiN.nodes()[n2]['Formula']
            if formA != formB:
                if formB in FDiN_knowledge.nodes():
                    if (formA, formB) in FDiN_knowledge.edges():
                        if (n1, n2) in FDiN.edges():
                            # Change edge weight if the edge already existed to 2
                            FDiN.edges()[(n1, n2)]['Weight'] = 2
                        else:
                            FDiN.add_edge(n1, n2, Transformation='Knowledge', Weight=2)
                    # Confirm nothing is being lost
                    elif (formB, formA) in FDiN_knowledge.edges():
                        print('-------')
print('Nº of edges in the FDiN before filtering:', len(FDiN.edges()))

# Filter FDiN to only include larger components
comps = []
for i in sorted(nx.connected_components(FDiN), key=len, reverse=True):
    if len(i) > 20:
        comps.extend(i)
FDiN = FDiN.subgraph(comps)
print('Nº of edges in the FDiN after filtering:', len(FDiN.edges()))

In [ ]:
len(FDiN.nodes())

In [ ]:
# save graph object to file
pickle.dump(FDiN, open('MD_FDiN_Final.pickle', 'wb'))

## Building the sample Formula-Difference Networks

For each sample, copy the FDiN and add the following node features:

- Peak Mass (divided by 100 so it has smaller values)
- (Treated) Intensity
- Feature Occurrence (1 or 0)

In [ ]:
sFDiNs_full = {}
for samp in sample_cols:

    sFDiNs_full[samp] = FDiN.copy()
    ints = {i: treated_data.loc[samp, i] for i in formula_df.index}
    pres = {i: bin_data.loc[samp, i] for i in formula_df.index}
    # Storing intensity of feature in sample, mass and node degree on the nodes
    intensity_attr = dict.fromkeys(sFDiNs_full[samp].nodes(),0)
    for m in sFDiNs_full[samp].nodes():
        intensity_attr[m] = {'mass':formula_df.loc[m,'Probable m/z']/100, 'intensity': ints[m], 'presence':pres[m]}
    nx.set_node_attributes(sFDiNs_full[samp], intensity_attr)

In [ ]:
# Description of the sFDiNs built
print('Full sMDiNs')
print('Number of nodes:', len(sFDiNs_full[samp].nodes()))
print('Number of edges:', len(sFDiNs_full[samp].edges()))

### Setting up the FDiGNN model

In [ ]:
# Node Features
print('Nº of node features excluding the string formula:',
      len(list(sFDiNs_full[samp].nodes()[list(sFDiNs_full[samp].nodes())[0]].keys())[1:]))
node_attrs = list(sFDiNs_full[samp].nodes()[list(sFDiNs_full[samp].nodes())[0]].keys())[1:]

# Edge Attributes
edge_attrs = list(sFDiNs_full[samp].edges()[list(sFDiNs_full[samp].edges())[0]].keys())
edge_attrs.remove('Transformation')

print('Nº of node features:', len(node_attrs))
print('Nº of edge features:', len(edge_attrs))

In [ ]:
# Convert the sFDiNs into PyTorch geometric
data_list_full = []
for samp in sFDiNs_full:
    pyg_graph = from_networkx(sFDiNs_full[samp],
                              group_node_attrs=list(sFDiNs_full[samp].nodes()[list(sFDiNs_full[samp].nodes())[0]].keys())[1:],
                              group_edge_attrs=edge_attrs)
    data_list_full.append(pyg_graph.to(device))
dataset = DataLoader(data_list_full)

# Adding target information to the sFDiNs
for g in range(len(target)):
    if target[g] == 'P.Vivax+Prior':
        data_list_full[g].y = torch.FloatTensor([1, 0, 0]).type(torch.LongTensor).to(device)
    elif target[g] == 'P.Vivax+NoPrior':
        data_list_full[g].y = torch.FloatTensor([0, 1, 0]).type(torch.LongTensor).to(device)
    else:
        data_list_full[g].y = torch.FloatTensor([0, 0, 1]).type(torch.LongTensor).to(device)

# Model

Model class, global attention pooling layer, training and testing functions

In [ ]:
# Settting up the model
class FDiGNN_TAG(torch.nn.Module):
    def __init__(self, hidden_channels, drop, n_node_feat, K, retrieve_steps=False):
        super(FDiGNN_TAG, self).__init__()
        torch.manual_seed(89356)
        self.conv1 = TAGConv(n_node_feat, hidden_channels, K=K)
        self.norm1 = BatchNorm1d(hidden_channels)
        self.conv2 = TAGConv(hidden_channels, hidden_channels, K=K)
        self.norm2 = BatchNorm1d(hidden_channels)
        self.conv3 = TAGConv(hidden_channels, hidden_channels, K=K)
        self.norm3 = BatchNorm1d(hidden_channels)
        self.conv4 = TAGConv(hidden_channels, hidden_channels, K=K)
        self.norm4 = BatchNorm1d(hidden_channels)
        #self.conv5 = TAGConv(hidden_channels, hidden_channels, K=K)
        #self.norm5 = BatchNorm1d(hidden_channels)
        self.pooling = GlobalAttentionPooling(hidden_channels)
        self.lin1 = Linear(hidden_channels, hidden_channels)
        #self.lin2 = Linear(hidden_channels, int(hidden_channels/2))
        self.lin3 = Linear(hidden_channels, 3)
        self.drop = drop
        self.last_att_conv1 = None
        self.last_att_conv2 = None
        self.last_att_conv3 = None
        self.leakyrelu1 = nn.LeakyReLU()
        self.leakyrelu2 = nn.LeakyReLU()
        self.leakyrelu3 = nn.LeakyReLU()
        self.leakyrelu4 = nn.LeakyReLU()
        #self.leakyrelu5 = nn.LeakyReLU()
        self.leakyrelu6 = nn.LeakyReLU()

    def forward(self, x, edge_index, batch, edge_weight, retrieve_steps=False):
        # 1. Obtain node embeddings 
        x1 = self.conv1(x, edge_index, edge_weight=edge_weight)
        x1_relu = self.leakyrelu1(x1)
        x1_norm = self.norm1(x1_relu)
        x1_drop = F.dropout(x1_norm, p=self.drop, training=self.training)
        x2 = self.conv2(x1_drop, edge_index, edge_weight=edge_weight)
        x2_relu = self.leakyrelu2(x2)
        x2_norm = self.norm2(x2_relu)
        x2_drop = F.dropout(x2_norm, p=self.drop, training=self.training)
        x3 = self.conv3(x2_drop, edge_index, edge_weight=edge_weight)
        x3_relu = self.leakyrelu3(x3)
        x3_norm = self.norm3(x3_relu)
        x3_drop = F.dropout(x3_norm, p=self.drop, training=self.training)
        x4 = self.conv4(x3_drop, edge_index, edge_weight=edge_weight)
        x4_relu = self.leakyrelu4(x4)
        x4_norm = self.norm4(x4_relu)
        x4_drop = F.dropout(x4_norm, p=self.drop, training=self.training)
        #x5 = self.conv5(x4_drop, edge_index, edge_weight=edge_weight)
        #x5_relu = self.leakyrelu5(x5)
        #x5_norm = self.norm5(x5_relu)
        #x5_drop = F.dropout(x5_norm, p=self.drop, training=self.training)

        # 2. Readout layer
        x_emb = self.pooling(x4_drop, batch)

        # 3. Apply a final classifier
        x_emb = F.dropout(x_emb, p=self.drop, training=self.training)
        x_emb = self.lin1(x_emb)
        x_emb = self.leakyrelu6(x_emb)
        x_emb = self.lin3(x_emb)
        if retrieve_steps:
            self.x = x
            self.x1 = x1
            self.x1_relu = x1_relu
            self.x1_norm = x1_norm
            self.x1_drop = x1_drop
            self.x2 = x2
            self.x2_relu = x2_relu
            self.x2_norm = x2_norm
            self.x2_drop = x2_drop
        return x_emb

class GlobalAttentionPooling(nn.Module):
    def __init__(self, in_channels):
        super(GlobalAttentionPooling, self).__init__()
        self.attention_nn = nn.Sequential(nn.Linear(in_channels, 1), nn.Sigmoid())
        self.sigmoid = nn.Sigmoid()
        self.last_scores = None
        self.x_weighted = None
        
    def forward(self, x, batch):
        scores = self.attention_nn(x).squeeze(-1)
        scores = softmax(scores, batch)
        x_weighted = x * scores.unsqueeze(-1)
        self.last_scores = scores
        self.x_weighted = x_weighted
        graph_embedding = scatter_add(x_weighted, batch, dim=0)
        
        return graph_embedding

    def get_attention_scores(self):
        return self.last_scores, self.x_weighted

# Functions to train and test the model
def train(model, train_loader, optimizer):
    model.train()
    losses = []
    grad_norms = []
    criterion = torch.nn.CrossEntropyLoss()
    for data in train_loader:  # Iterate in batches over the training dataset.
        out = model(data.x.float(), data.edge_index, data.batch, data.edge_attr.float(), True)  # Perform a single forward pass.
        loss = criterion(out.cpu(), data.y.reshape(data.batch_size, 3).type(torch.FloatTensor))  # Compute the loss.
        loss.backward()  # Derive gradients.
        losses.append(loss.to('cpu').detach().numpy())
        optimizer.step()  # Update parameters based on gradients.
        optimizer.zero_grad()  # Clear gradients.
    return np.mean(losses), grad_norms, model

def test(model, loader):
    model.eval()

    correct = 0
    losses = []
    criterion = torch.nn.CrossEntropyLoss()
    for data in loader:  # Iterate in batches over the training/test dataset.
        out = model(data.x.float(), data.edge_index, data.batch, data.edge_attr.float(), True)  
        pred = out.argmax(dim=1)  # Use the class with highest probability.
        correct += int((pred.cpu() == data.y.reshape(data.batch_size, 3).type(torch.FloatTensor).argmax(dim=1)).sum())
        loss = criterion(out.cpu(), data.y.reshape(data.batch_size, 3).type(torch.FloatTensor))  # Compute the loss.
        losses.append(loss.to('cpu').detach().numpy())
    return (correct / len(loader.dataset), np.mean(losses), out)  # Derive ratio of correct predictions.

## Estimating Model Performance

Average of 10 iterations of the train/test splits.

**For this dataset this was performed in a separate file. On the cell below we copy the relevant part of that file here, adjusting it to this notebook. Non-relevant parts of the code were removed.**

**This will take a lot of time, put False to skip. Models are already pre-saved.**

In [ ]:
# Put True if you want to fit new models, else put False
fit_models = False

In [ ]:
np.random.seed(174)

# Setting parameters
max_epochs = 500

# Setting up store results
save_models_all = {}

# Setting up store results
loss_dict = {}
train_accuracy_dict = {}
test_accuracy_dict = {}

classes = pd.unique(pd.Series(target))

print('Starting model fitting.')

# For each repetition
for r in range(iter_num):

    train_index = [treated_data.index.get_loc(i) for i in train_idxs[r+1]]
    test_index = [treated_data.index.get_loc(i) for i in test_idxs[r+1]]

    train_loader = DataLoader(pd.Series(data_list_full)[train_index].values, batch_size=16, shuffle=True)
    test_loader = DataLoader(pd.Series(data_list_full)[test_index].values, batch_size=32, shuffle=False)

    # Setting up the models
    model = FDiGNN_TAG(hidden_channels=64, drop=0.3, n_node_feat=len(node_attrs), K=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
    criterion = torch.nn.CrossEntropyLoss()

    # Temporary store lists
    loss_list = []
    train_accuracy_list = []
    test_accuracy_list = []

    patience_counter = 0
    best_loss = 10
    
    if fit_models:

        # Train the model
        for epoch in range(1, max_epochs+1):
            model.float()
            loss, g_norm, _ = train(model, train_loader, optimizer)
            train_acc, _, _ = test(model, train_loader)
            test_acc, _, _ = test(model, test_loader)
            if epoch%20 == 0:
                print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}, Learning Rate:{scheduler.get_last_lr()}')
            loss_list.append(loss)
            train_accuracy_list.append(train_acc)
            test_accuracy_list.append(test_acc)
            scheduler.step()

            if loss < 0.9*best_loss:
                best_loss = loss
                patience_counter = 0
            else:
                patience_counter += 1

            #if patience_counter >= max_patience_counter and epoch>100:
            #    break

        # Store results
        loss_dict[r] = loss_list
        train_accuracy_dict[r] = train_accuracy_list
        test_accuracy_dict[r] = test_accuracy_list
        # Store results
        print(f'Number of parameters: {sum(p.numel() for p in model.parameters())}.')
        print(f'Model Accuracy: {test_accuracy_list[-1]*100:.2f} %')
        print('----------------------------------------')
        print('\n')
        torch.save(model.state_dict(), f'Models/MD_model_4TAG_64HC3D005LR0001WD3K_{r}')

    else:
        # Load the model
        model = FDiGNN_TAG(hidden_channels=64, drop=0.3, n_node_feat=len(node_attrs), K=3).to(device)

        # Load the model weights
        model.load_state_dict(torch.load(f'Models/MD_model_4TAG_64HC3D005LR0001WD3K_{r}'))

        test_acc, _, _ = test(model, test_loader)
        test_accuracy_list = [test_acc,]
        test_accuracy_dict[r] = test_accuracy_list
        print(f'Model Accuracy: {test_accuracy_list[-1]*100:.2f} %')
        print('----------------------------------------')

In [ ]:
# Store results
print('MD - 4TAG_AttPool_2Class')
print(f'Number of parameters: {sum(p.numel() for p in model.parameters())}.')
print('-----')
print('Final Acc.', pd.DataFrame(test_accuracy_dict).iloc[-1].mean(), '+-',
      pd.DataFrame(test_accuracy_dict).iloc[-1].std())

## Fitting the Model

Since we are using the same train/test split conditions to extract node importance considering the test samples only, we will read the first model fitted by the first train/test split performed instead of fitting a new one.

In [ ]:
from torchinfo import summary

# Set up model
model = FDiGNN_TAG(hidden_channels=64, drop=0.3, n_node_feat=len(node_attrs), K=3).to(device)

# Load the model weights
model.load_state_dict(torch.load('Models/MD_model_4TAG_64HC3D005LR0001WD3K_0'))

# Summary of the model
summary(model)

In [ ]:
# Confirm the accuracy of the model is as supposed to be
train_index = [treated_data.index.get_loc(i) for i in train_idxs[1]]
test_index = [treated_data.index.get_loc(i) for i in test_idxs[1]]

train_loader = DataLoader(pd.Series(data_list_full)[train_index].values, batch_size=16, shuffle=True)
test_loader = DataLoader(pd.Series(data_list_full)[test_index].values, batch_size=32, shuffle=False)

test_acc, _, _ = test(model, test_loader)
test_acc

## Node Importance

Get Node Importance - This was simplified to only get the prediction impact node importance measure since it was the one used in the paper. The other strategies were removed mainly the entropy integral strategy. They were removed since they all provided similar results and from the methods the prediction impact one is the most straightforward one.

**Only the test samples of the train/test split are used to calculate the prediction impact change of each node/metabolite.**

In [ ]:
effect = {}
all_preds ={}


train_index = [treated_data.index.get_loc(i) for i in train_idxs[1]]
test_index = [treated_data.index.get_loc(i) for i in test_idxs[1]]

# Normal Predictions in unchanged samples
out_normal = pd.DataFrame()
test_samples = pd.Series(data_list_full)[test_index].values
test_loader = DataLoader(pd.Series(data_list_full)[test_index].values, batch_size=32, shuffle=False)
for data in test_loader:  # Iterate in batches over the training/test dataset.
    model.eval()
    out = model(data.x.float(), data.edge_index, data.batch, data.edge_attr.float(), retrieve_steps=True)
    out = F.softmax(out, 1)
    out_normal = pd.concat((out_normal, pd.DataFrame(out.detach().cpu().numpy())))
all_preds['Normal'] = out_normal.reset_index().iloc[:,1:].to_dict()

# For each metabolite/node
for i in tqdm(range(len(FDiN.nodes()))):
    # Get node index, unchanged values and seet the quantile values
    node = list(FDiN.nodes())[i]
    original_values = treated_data.loc[test_idxs[1], node].copy().values
    original_feature_values = bin_data.loc[test_idxs[1], node].copy().values
    q_values = [0.05, 0.5, 0.95]

    # Set the stores
    all_preds[node] = {}
    effect[i] = pd.DataFrame(columns=q_values)
    # Get the quantiles
    quantile_values = np.quantile(original_values, q=q_values)
    quantile_feature_values = np.quantile(original_feature_values, q=q_values)

    for q in range(len(quantile_values)):
        # Change all samples for the current quantile value
        for g in range(len(test_samples)):
            test_samples[g].x[i, 1] = quantile_values[q]
            if quantile_feature_values[q] != 0:
                if quantile_feature_values[q] != 1:
                    test_samples[g].x[i, 2] = 1
                else:
                    test_samples[g].x[i, 2] = quantile_feature_values[q]
            else:
                test_samples[g].x[i, 2] = quantile_feature_values[q]

        # See and store model predictions of changed samples
        out_shuffled = pd.DataFrame()
        test_loader = DataLoader(test_samples, batch_size=32, shuffle=False)
        for data in test_loader:  # Iterate in batches over the training/test dataset.
            model.eval()
            out = model(data.x.float(), data.edge_index, data.batch, data.edge_attr.float(), retrieve_steps=True)
            out = F.softmax(out, 1)
            out_shuffled = pd.concat((out_shuffled, pd.DataFrame(out.detach().cpu().numpy())))
        all_preds[node][q] = out_shuffled.reset_index().iloc[:,1:].to_dict()

        # Calculate the absolute prediction differences and store them
        results = pd.DataFrame((out_normal - out_shuffled)).abs()
        effect[i][q_values[q]] = results.max(axis=1)

    # Restore values
    for g in range(len(test_samples)):
        test_samples[g].x[i, 1] = original_values[g]
        test_samples[g].x[i, 2] = original_feature_values[g]

In [ ]:
# Join the results from the different nodes, quantiles and samples and ranking the nodes
new_df = pd.DataFrame(columns=range(len(test_idxs[1])))
for node in effect.keys():
    # Get the maximum values for each quantile for each node/sample pair
    new_df.loc[node] = effect[node].max(axis=1).values
new_df.index = list(FDiN.nodes())

# Normalize all predictions changes by sample - make each sample have teh same weight for node importance calculation
new_df = (new_df/new_df.sum()).replace({np.nan:0})
global_effect = new_df.T.apply(
    lambda x: x.sort_values(ascending=False).values).T

# Get the median of all normalzied prediction changes across the samples
global_effect = global_effect.median(axis=1).sort_values(ascending=False)

In [ ]:
global_effect.head(20)

# Supervised Analysis Filtering to the Nodes in the FDiN

### Random Forest

In [ ]:
# Choose a number for the seed for consistent results
np.random.seed(65824802)
n_trees=200 # Number of trees in the model

RF_accus = {}
RF_imp_feats = {}

for a in train_idxs:
    RF_model = metsta.RF_model(treated_data.loc[train_idxs[a], list(FDiN.nodes())],
                    train_tg[a], regres=False, # Data, labels and if it's a regression or classification
                    return_cv=False,
                    n_trees=n_trees, # Number of trees in the model
                    # Choose a method of cross-validation (None is stratified cv) and the number of folds
             metrics = ('accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted')) # Choose the performance metric

    rf_preds = RF_model.predict(treated_data.loc[test_idxs[a], list(FDiN.nodes())])
    correct = 0
    for i in range(len(rf_preds)):
        if rf_preds[i] == test_tg[a][i]:
            correct += 1
    accuracy = correct/len(test_idxs[a])
    print(f'{a}:', accuracy)

    RF_accus[a] = accuracy
    RF_imp_feats[a] = RF_model.feature_importances_
    print(f'Finished fitting Random Forests for {a}.')

In [ ]:
imp_feat_sum = pd.DataFrame(RF_imp_feats).sum(axis=1)/len(RF_imp_feats)
sorted_imp_feat = sorted(enumerate(imp_feat_sum), key=lambda x: x[1], reverse=True)

imp_feats_rf = processed_data[['Probable m/z']].loc[list(FDiN.nodes())].copy()
imp_feats_rf.insert(0,'Bucket label', imp_feats_rf.index)
imp_feats_rf.insert(1,'Gini', '')
for n in range(len(sorted_imp_feat)):
    imp_feats_rf.loc[treated_data.loc[:, list(FDiN.nodes())].columns[sorted_imp_feat[n][0]],
                     'Gini'] = sorted_imp_feat[n][1]
rank_imp_feats_rf = imp_feats_rf.rank(ascending=False)
rank_imp_feats_rf.sort_values(by='Gini')

## PLS-DA

In [ ]:
%%capture --no-stdout
# above is to supress PLS warnings

# Set the random seed
np.random.seed()

max_comp = 20 # Max. number of components to search (the higher the more time it takes)

# Store Results
PLS_optim = metsta.optim_PLSDA_n_components(treated_data.loc[:, list(FDiN.nodes())], target, regres=False, # Data, target and if it's a regression
                                    encode2as1vector=True,
                                    max_comp=max_comp, # Max. number of components to search
                                    kf=None, n_fold=5, # Cross validation to use (none is stratified CV) and nº of folds
                                    scale=False) # Set scale to True only if you did not do scaling in pre-treatments

In [ ]:
scores_cols = sns.color_palette('tab10', 10) # Set the colors for the lines
with sns.axes_style("whitegrid"):
    with sns.plotting_context("notebook", font_scale=1.2):
        f, ax = plt.subplots(1, 1, figsize=(5,5), constrained_layout=True) # Set the figure size
        c = 0
        for i, values in PLS_optim.items():
            if i =='CVscores':
                name = 'Q$^2$'
            else:
                name = 'R$^2$'
            
            ax.plot(range(1, len(values) + 1), values, label=name, color = scores_cols[c])
            c = c+1
        
        ax.set(xlabel='Number of Components', # Set the label for the x axis
                ylabel='PLS Score') # Set the label for the Y axis
        ax.legend(loc='lower right', fontsize=15) # Set the legend
        ax.set_ylim([0, 1.02]) # Set limits for y axis
        ax.set_xticks(range(0, len(values), 2)) # Set ticks that appear in the bottom of x axis
        plt.show()

PLS-DA Model Fitting

In [ ]:
%%capture --no-stdout
# above is to supress PLS warnings
# Choose a number for the seed for consistent results
np.random.seed(65824802)

n_comp = 16 # Number of components of PLS-DA model - very important

PLSDA_accus = {}
PLSDA_imp_feats = {}

for a in train_idxs:

    matrix_train = _generate_y_PLSDA(train_tg[a], pd.unique(target), False)
    matrix_test = _generate_y_PLSDA(test_tg[a], pd.unique(target), False)

    plsda = PLSRegression(n_components=n_comp, scale=False)
    # Fit PLS model
    plsda.fit(X=treated_data.loc[train_idxs[a], list(FDiN.nodes())], Y=matrix_train)
    # Obtain results with the test group
    y_pred = plsda.predict(treated_data.loc[test_idxs[a], list(FDiN.nodes())])

    accuracy = (matrix_test.idxmax(axis=1) == pd.DataFrame(
        y_pred, columns=matrix_test.columns).idxmax(axis=1)).sum()/len(matrix_test)

    PLSDA_accus[a] = accuracy
    PLSDA_imp_feats[a] = _calculate_vips(plsda)
    print(f'{a}:', accuracy)
    print(f'Finished fitting PLS-DA for {a}.')

In [ ]:
imp_feat_sum = pd.DataFrame(PLSDA_imp_feats).sum(axis=1)/len(PLSDA_imp_feats)
sorted_imp_feat = sorted(enumerate(imp_feat_sum), key=lambda x: x[1], reverse=True)

imp_feats_plsda = processed_data[['Probable m/z']].loc[list(FDiN.nodes())].copy()
imp_feats_plsda.insert(0,'Bucket label', imp_feats_plsda.index)
imp_feats_plsda.insert(1,'VIP', '')
for n in range(len(sorted_imp_feat)):
    imp_feats_plsda.loc[treated_data.loc[:, list(FDiN.nodes())].columns[sorted_imp_feat[n][0]],
            'VIP'] = sorted_imp_feat[n][1]
rank_imp_feats_plsda = imp_feats_plsda.rank(ascending=False)
rank_imp_feats_plsda.sort_values(by='VIP')

# Comparing Important Metabolites

We compared the rankings of important metabolites between RF, PLS-DA and FDiGNN models in two ways:

- The commonality of important metabolites between the models
- The number of edges established between the important metabolites in the FDiN

In [ ]:
all_ranks = pd.DataFrame(global_effect).rank(ascending=False).sort_values(by=0)
all_ranks

In [ ]:
np.random.seed(492043)

# Store results
intersections = {}
intersections['PLS-DA - RF'] = []
intersections['PLS-DA - FDiGNN'] = []
intersections['RF - FDiGNN'] = []
random_intersections = []

# Shuffle the list of metabolites
idxs1 = list(rank_imp_feats_plsda.index)
np.random.shuffle(idxs1)

# For each top X metabolites see how many metabolites are in common between the methods and to a random assortment of peaks
for i in range(len(rank_imp_feats_plsda)):
    plsda_r = rank_imp_feats_plsda.sort_values(by='VIP').index[:i]
    rf_r = rank_imp_feats_rf.sort_values(by='Gini').index[:i]
    gnn_r = all_ranks.sort_values(by=0).index[:i]

    intersections['PLS-DA - RF'].append(np.intersect1d(plsda_r, rf_r)) 
    intersections['PLS-DA - FDiGNN'].append(np.intersect1d(plsda_r, gnn_r)) 
    intersections['RF - FDiGNN'].append(np.intersect1d(rf_r, gnn_r))
    random_intersections.append(np.intersect1d(idxs1[:i], gnn_r))

In [ ]:
# Calculate the number of edges established between the the important metabolites in the FDiN for each model
edge_numbers = {'PLS-DA': [], 'RF': [], 'FDiGNN': [], 'Random': []}
for i in range(len(rank_imp_feats_plsda)):
    plsda_r = rank_imp_feats_plsda.sort_values(by='VIP').index[:i]
    rf_r = rank_imp_feats_rf.sort_values(by='Gini').index[:i]
    pred = all_ranks.sort_values(by=0).index[:i]

    edge_numbers['PLS-DA'].append(len(FDiN.subgraph(plsda_r).edges()))
    edge_numbers['RF'].append(len(FDiN.subgraph(rf_r).edges()))
    edge_numbers['FDiGNN'].append(len(FDiN.subgraph(pred).edges()))
    edge_numbers['Random'].append(len(FDiN.subgraph(idxs1[:i]).edges()))

Plot the figure

**For Figure 4.**

In [ ]:
f,(axl,axc, axr) = plt.subplots(1,3, figsize=(18,6), constrained_layout=True)

for a in intersections:
    axl.plot([(i+1)/(len(intersections[a])) for i in range(len(intersections[a]))],
             [len(i)/(len(intersections[a])) for i in intersections[a]], label=a)
    percentage = []
    for i in range(len(intersections[a])):
        percentage.append(len(intersections[a][i])/(i+1))
    axc.plot([(i+1)/(len(intersections[a])) for i in range(len(intersections[a]))], percentage, label=a)
axl.plot([(i+1)/(len(random_intersections)) for i in range(len(random_intersections))],
         [len(i)/(len(random_intersections)) for i in random_intersections], label='Random')
percentage = []
for i in range(len(random_intersections)):
    percentage.append(len(random_intersections[i])/(i+1))
axc.plot([(i+1)/(len(random_intersections)) for i in range(len(random_intersections))], percentage)
axl.legend(fontsize=16)
f.suptitle('MD', fontsize=25)
axl.set_ylabel('Fraction of Total Peaks in Common', fontsize=16)
axc.set_ylabel('Fraction of Peaks in Common', fontsize=16)
axl.set_xlabel('Fraction of Top Important Features Considered', fontsize=16)
axc.set_xlabel('Fraction of Top Important Features Considered', fontsize=16)

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Create inset
axins = inset_axes(axr, width="25%", height="35%", loc='lower right', borderpad=1.7)

for a in edge_numbers:
    axr.plot([(i+1)/(len(edge_numbers[a])) for i in range(len(edge_numbers[a]))], edge_numbers[a], label=a)
    axins.plot([(i+1)/(len(edge_numbers[a])) for i in range(len(edge_numbers[a]))], edge_numbers[a], label=a)
axr.legend(fontsize=16)
axr.set_ylabel('Edges Established between Imp. Features', fontsize=14)
axr.set_xlabel('Fraction of Top Important Features Considered', fontsize=16)

axins.set_xlim([0,0.15])
axins.set_ylim([0,300])

# Save figure
f.savefig('Paper_Figs/MD_CommonImpFeats.png', dpi=600)
f.savefig('Paper_Figs/MD_CommonImpFeats.svg', dpi=600)

Correlation Analysis

In [ ]:
# FDiGNN - Correlation between closeness centrality and FDiGNN prediction impact
together = pd.concat((global_effect.rank(ascending=False), pd.Series(nx.closeness_centrality(FDiN))), axis=1)
stats.spearmanr(together[0], together[1])

In [ ]:
plt.scatter(together[0], together[1], s=2)

In [ ]:
# PLS-DA - Correlation between closeness centrality and VIP scores
together = pd.concat((rank_imp_feats_plsda['VIP'], pd.Series(nx.closeness_centrality(FDiN))), axis=1)
stats.spearmanr(together['VIP'], together[0])

In [ ]:
# RF - Correlation between closeness centrality and Gini Importance
together = pd.concat((rank_imp_feats_rf['Gini'], pd.Series(nx.closeness_centrality(FDiN))), axis=1)
stats.spearmanr(together['Gini'], together[0])

In [ ]:
# FDiGNN - Correlation between Degree and FDiGNN prediction impact
together = pd.concat((global_effect.rank(ascending=False), pd.Series(dict(nx.degree(FDiN)))), axis=1)
stats.spearmanr(together[0], together[1])

In [ ]:
# FDiGNN - Correlation between Betweenness Centrality and FDiGNN prediction impact
together = pd.concat((global_effect.rank(ascending=False), pd.Series(nx.betweenness_centrality(FDiN))), axis=1)
stats.spearmanr(together[0], together[1])

# Analysing GNN Important Metabolites

In [ ]:
# Base selection of top metabolites to see (in % - 0.1 is the top 10% of metabolites)
base_threshold_of_importance = 0.02
base_threshold_of_importance2 = 0.25

In [ ]:
# Setting FDiGNN and PLS-DA rank as attributes
nx.set_node_attributes(FDiN, dict(all_ranks[0]), name='FDiGNN Rank')
nx.set_node_attributes(FDiN, dict(rank_imp_feats_plsda['VIP']), name='VIP (PLS-DA) Rank')

## Pathway Enrichment Analysis

Very similar pathways repeated many times were merged to a single pathway:

- 22657 pathway names starting with 'De Novo Triacylglycerol Biosynthesis' merged to 'De Novo Triacylglycerol Biosynthesis'
- 923 pathway names starting with 'Phosphatidylethanolamine Biosynthesis' merged to 'Phosphatidylethanolamine Biosynthesis'
- 923 pathway names starting with 'Phosphatidylcholine Biosynthesis' merged to 'Phosphatidylcholine Biosynthesis'
- 3278 pathway names starting with 'Cardiolipin Biosynthesis' merged to 'Cardiolipin Biosynthesis'

In [ ]:
for node in FDiN_knowledge.nodes():
    n_p = []
    for p in FDiN_knowledge.nodes()[node]['Pathways']:
        if p.startswith('De Novo Triacylglycerol Biosynthesis'):
            if 'De Novo Triacylglycerol Biosynthesis' not in n_p:
                n_p.append('De Novo Triacylglycerol Biosynthesis')
        elif p.startswith('Phosphatidylethanolamine Biosynthesis'):
            if 'Phosphatidylethanolamine Biosynthesis' not in n_p:
                n_p.append('Phosphatidylethanolamine Biosynthesis')
        elif p.startswith('Phosphatidylcholine Biosynthesis'):
            if 'Phosphatidylcholine Biosynthesis' not in n_p:
                n_p.append('Phosphatidylcholine Biosynthesis')
        elif p.startswith('Cardiolipin Biosynthesis'):
            if 'Cardiolipin Biosynthesis' not in n_p:
                n_p.append('Cardiolipin Biosynthesis')
        else:
            n_p.append(p)
    FDiN_knowledge.nodes()[node]['Pathways'] = n_p

for edge in FDiN_knowledge.edges():
    n_p = []
    for p in FDiN_knowledge.edges()[edge]['Pathways']:
        if p.startswith('De Novo Triacylglycerol Biosynthesis'):
            if 'De Novo Triacylglycerol Biosynthesis' not in n_p:
                n_p.append('De Novo Triacylglycerol Biosynthesis')
        elif p.startswith('Phosphatidylethanolamine Biosynthesis'):
            if 'Phosphatidylethanolamine Biosynthesis' not in n_p:
                n_p.append('Phosphatidylethanolamine Biosynthesis')
        elif p.startswith('Phosphatidylcholine Biosynthesis'):
            if 'Phosphatidylcholine Biosynthesis' not in n_p:
                n_p.append('Phosphatidylcholine Biosynthesis')
        elif p.startswith('Cardiolipin Biosynthesis'):
            if 'Cardiolipin Biosynthesis' not in n_p:
                n_p.append('Cardiolipin Biosynthesis')
        else:
            n_p.append(p)
    FDiN_knowledge.edges()[edge]['Pathways'] = n_p

Use the metabolic knowledge network to associate metabolic pathways to nodes (if the formula is in the pathway) and to edges (if both connecting formulas are in the pathway)

In [ ]:
# For Nodes
formulas_in_knowledge = list(FDiN_knowledge.nodes())
pathway_ranks = {}
pathway_series = {}
nodes_in_paths = {}
for node in FDiN.nodes():
    curr_formula = FDiN.nodes()[node]['Formula']
    if isinstance(processed_data.loc[node, 'Matched HMDB names'], list):
        FDiN.nodes()[node]['Name'] = processed_data.loc[node, 'Matched HMDB names']
        FDiN.nodes()[node]['HMDB_ID'] = processed_data.loc[node, 'Matched HMDB IDs']
    else:
        FDiN.nodes()[node]['Name'] = 'None'
        FDiN.nodes()[node]['HMDB_ID'] = 'None'

    if curr_formula in formulas_in_knowledge:
        nodes_in_paths[node] = pd.Series()
        FDiN.nodes()[node]['Names in Pathways'] = FDiN_knowledge.nodes()[curr_formula]['Names']
        FDiN.nodes()[node]['HMDB_ID in Pathways'] = FDiN_knowledge.nodes()[curr_formula]['HMDB_ID']
        FDiN.nodes()[node]['Pathways'] = FDiN_knowledge.nodes()[curr_formula]['Pathways']
        FDiN.nodes()[node]['SMPDB_IDs'] = FDiN_knowledge.nodes()[curr_formula]['SMPDB_IDs']
        for p in FDiN_knowledge.nodes()[curr_formula]['Pathways']:
            if p in pathway_ranks:
                pathway_ranks[p].append(all_ranks.loc[node, 0])
                pathway_series[p].loc[node] = all_ranks.loc[node, 0]
            else:
                pathway_ranks[p] = [all_ranks.loc[node, 0],]
                pathway_series[p] = pd.Series()
                pathway_series[p].loc[node] = all_ranks.loc[node, 0]
            nodes_in_paths[node].loc[p] = 1
            
    else:
        FDiN.nodes()[node]['Names in Pathways'] = 'None'
        FDiN.nodes()[node]['HMDB_ID in Pathways'] = 'None'
        FDiN.nodes()[node]['Pathways'] = 'None'
        FDiN.nodes()[node]['SMPDB_IDs'] = 'None'

In [ ]:
# For Edges
for edge in FDiN.edges():
    formA = FDiN.nodes()[edge[0]]['Formula']
    formB = FDiN.nodes()[edge[1]]['Formula']
    if (formA, formB) in FDiN_knowledge.edges():
        FDiN.edges()[edge]['Pathways'] = FDiN_knowledge.edges()[(formA, formB)]['Pathways']
        FDiN.edges()[edge]['SMPDB_IDs'] = FDiN_knowledge.edges()[(formA, formB)]['SMPDB_IDs']
    else:
        FDiN.edges()[edge]['Pathways'] = 'None'
        FDiN.edges()[edge]['SMPDB_IDs'] = 'None'

In [ ]:
# Get only the nodes with associated pathways and their ranks
# Then make a dict to re-order those ranks skipping nodes without associated pathways
s = pd.DataFrame(pathway_series).values.flatten()
s = pd.unique(pd.Series(s[~np.isnan(s)]).sort_values())
path_ranks_short = dict(zip(s,range(1, len(s)+1)))

# Use the dictionary made to get the ranks per metabolic pathway after this association
for i in pathway_ranks:
    pathway_ranks[i].sort()
pathway_ranks
for p in pathway_ranks:
    pathway_ranks[p] = [path_ranks_short[i] for i in pathway_ranks[p]]
pathway_ranks

For pathway enrichment analysis, consider only pathways with at least 3 detected compounds in the FDiN and that establish at least 1 edge between the FDiNs.

In [ ]:
paths = pd.Series(nx.get_node_attributes(FDiN, 'Pathways')).explode().value_counts()
paths.pop('None')
paths = paths[paths>2]

paths_edges = pd.Series(nx.get_edge_attributes(FDiN, 'Pathways')).explode().value_counts()
paths_edges.pop('None')
paths = paths[[p for p in paths.index if p in paths_edges.index]]
paths_edges = paths_edges[paths.index]

In [ ]:
# 'Significant' Metabolites - Number of Metabolites in the top ranks considered with at least one associated pathway
top_ranks_considered = (
    np.array(list(path_ranks_short.keys())) <= int(base_threshold_of_importance * len(FDiN.nodes()))).sum()
annotated_metabolites_w_path = top_ranks_considered

# Background Set - Number of Metabolites detected in the FDiN with at least one associated pathway
total_metabolites_with_pathways = len(path_ranks_short)

# Get the counts of metabolites of each pathway
path_assign_hmdb = paths

# Preparing DF
over_representation_df = pd.DataFrame(columns=['Pathway Name',
    'Nº of Met. in Dataset', 'Nº of Met. in Pathway', '% of Met. In Set', 'Probability'], dtype='object')

for pathway in tqdm(path_assign_hmdb.index):
    # Pathway Name
    p_name = pathway
    # Pathway Background Set - Number of Metabolites detected in the FDiN in each pathway
    total_met_in_path = path_assign_hmdb.loc[pathway]
    # 'Significant' Metabolites in Pathway - Number of Metabolites in the top ranks considered in each pathway
    ann_met_in_path = (np.array(pathway_ranks[pathway]) <= top_ranks_considered).sum()

    # Calculating probability to find ann_met_in_path or more metabolites in annotated_metabolites_w_path
    prob = stats.hypergeom(M=total_metabolites_with_pathways, 
                            n=total_met_in_path, 
                            N=annotated_metabolites_w_path).sf(ann_met_in_path-1)

    # Adding the line to the DF
    over_representation_df.loc[pathway] = [p_name, ann_met_in_path, total_met_in_path,
                                           ann_met_in_path/total_met_in_path*100, prob]

# Sorting DataFrame from least probable to more probable and adding adjusted probability
# with Benjamini-Hochberg multiple test correction
over_representation_df_nodes = over_representation_df.sort_values(by='Probability')
over_representation_df_nodes['Adjusted (BH) Probability'] = metsta.p_adjust_bh(over_representation_df_nodes['Probability'])

#### For Supplementary Table 6

In [ ]:
over_representation_df_nodes.iloc[:, 1:]

In [ ]:
# 'Significant' Metabolites - Number of Metabolites in the top ranks considered with at least one associated pathway
top_ranks_considered = (
    np.array(list(path_ranks_short.keys())) <= int(base_threshold_of_importance2 * len(FDiN.nodes()))).sum()
annotated_metabolites_w_path = top_ranks_considered

# Background Set - Number of Metabolites detected in the FDiN with at least one associated pathway
total_metabolites_with_pathways = len(path_ranks_short)

# Get the counts of metabolites of each pathway
path_assign_hmdb = paths

# Preparing DF
over_representation_df = pd.DataFrame(columns=['Pathway Name',
    'Nº of Met. in Dataset', 'Nº of Met. in Pathway', '% of Met. In Set', 'Probability'], dtype='object')

for pathway in tqdm(path_assign_hmdb.index):
    # Pathway Name
    p_name = pathway
    # Pathway Background Set - Number of Metabolites detected in the FDiN in each pathway
    total_met_in_path = path_assign_hmdb.loc[pathway]
    # 'Significant' Metabolites in Pathway - Number of Metabolites in the top ranks considered in each pathway
    ann_met_in_path = (np.array(pathway_ranks[pathway]) <= top_ranks_considered).sum()

    # Calculating probability to find ann_met_in_path or more metabolites in annotated_metabolites_w_path
    prob = stats.hypergeom(M=total_metabolites_with_pathways, 
                            n=total_met_in_path, 
                            N=annotated_metabolites_w_path).sf(ann_met_in_path-1)

    # Adding the line to the DF
    over_representation_df.loc[pathway] = [p_name, ann_met_in_path, total_met_in_path,
                                           ann_met_in_path/total_met_in_path*100, prob]

# Sorting DataFrame from least probable to more probable and adding adjusted probability
# with Benjamini-Hochberg multiple test correction
over_representation_df_nodes = over_representation_df.sort_values(by='Probability')
over_representation_df_nodes['Adjusted (BH) Probability'] = metsta.p_adjust_bh(over_representation_df_nodes['Probability'])

In [ ]:
over_representation_df_nodes.iloc[:, 1:]

In [ ]:
# Ranking edges as the mean of the ranks of the nodes it links
edges = pd.Series(nx.get_edge_attributes(FDiN, 'Pathways'))#.explode()
edge_rank = {}
for e1, e2 in edges.index:
    edge_rank[(e1, e2)] = np.mean([all_ranks.loc[e1].values[0], all_ranks.loc[e2].values[0]])
pd.Series(edge_rank).sort_values()
edge_rank_rank = pd.Series(edge_rank)
edge_rank_rank

In [ ]:
# Get the median edge rank of the edges associated with each pathway
short_edges = edges[edges != 'None']
short_edges = short_edges.explode()
path_edge_medians = {}
for i in paths.index:
    edges_p = short_edges[short_edges == i]
    path_edge_medians[i] = edge_rank_rank.loc[edges_p.index].median()

## Dash App For Highlighting Important Network Areas

**For Figures 8A and B.**

In [ ]:
classes = pd.unique(pd.Series(target))
# Get the list of sorted importances to use to map
sorted_imps = (global_effect).sort_values(ascending=False)
sorted_imps

In [ ]:
FDiN_knowledge.nodes()
a= []
in_knowledge = {}
for i in FDiN.nodes():
    if FDiN.nodes()[i]['Formula'] in FDiN_knowledge.nodes():
        in_knowledge[i] = {'In Knowledge': True}
        a.append(FDiN.nodes()[i]['Formula'])
    else:
        in_knowledge[i] = {'In Knowledge': False}
        
#len(a), len(FDiN_knowledge.nodes()), in_knowledge

In [ ]:
nx.set_node_attributes(FDiN, in_knowledge)
#FDiN.nodes()['511.3533_453.5']
#[i for i in FDiN_knowledge.nodes() if i not in a]

In [ ]:
# save graph object to file
#pickle.dump(FDiN, open('MD_FDiN_AllAtributes_ForDash.pickle', 'wb'))

In [ ]:
# Load Extra layouts for saving svg files
cyto.load_extra_layouts()

# Transform FDiN into Cytoscape data
cs_data = nx.cytoscape_data(FDiN)
elements = cs_data["elements"]["nodes"] + cs_data["elements"]["edges"]

# Make a 'graph' to act as node legend
graph_legend = nx.Graph()
graph_legend.add_nodes_from(['Pathway Node','Rel. Near Pathway','Non-Pathway Node'])
nx.set_node_attributes(graph_legend, {'Pathway Node':{'color':'Red'},
                                       'Rel. Near Pathway':{'color':'lightcoral'},
                                       'Non-Pathway Node':{'color':'lightgrey'}})
cs_legend = nx.cytoscape_data(graph_legend)
for n, p in zip(cs_legend["elements"]["nodes"], range(0,250,50)):
    n["position"] = {"x": -200, "y": p}
elements_legend = cs_legend["elements"]["nodes"] + cs_legend["elements"]["edges"]

# Get an initial order of pathways
rel_paths = over_representation_df_nodes.index

# Get specific positions for preset layout of the network
pos = nx.kamada_kawai_layout(FDiN)
for n1, p in zip(cs_data["elements"]["nodes"], pos.values()):
    n1["position"] = {"x": p[0] * 2000, "y": p[1] * 2000}

# Initiate the App
app = Dash()

# Dropdown menu for network layout
dropdown_layout = dcc.Dropdown(
    id='dropdown-update-layout',
    value='preset',
    clearable=False,
    options=[
        {'label': name.capitalize(), 'value': name}
        for name in ['preset', 'grid', 'random', 'circle', 'cose', 'concentric']
    ]
)

# Dropdown menu for pathway chosen
dropdown_layout2 = dcc.Dropdown(
    id='dropdown-update-layout2',
    value='None',
    clearable=False,
    options=[
        {'label': name.capitalize(), 'value': name}
        for name in ['None',] + list(rel_paths)
    ]
)

# Slider to select the threshold to use to consider important metabolites
top_mets_chosen = dcc.Slider(0, 0.5, 0.005, value=0.05, id='top_mets_chosen',
                            marks={0: {'label': '0'}, 0.02: {'label': '0.02'}, 0.05: {'label': '0.05'}, 
                                   0.1: {'label': '0.10'}, 0.2: {'label': '0.20'}, 0.5: {'label': '0.50'} })

# Organize the APP layout
app.layout = html.Div([html.Div([# Initial Options
                       dcc.Markdown('Graph Layout:',),
                       dropdown_layout,
                       dcc.Markdown('Pathway Highlighted:',),
                       dropdown_layout2,
                       dcc.Markdown('Nº of Important Metabolites:',),
                       top_mets_chosen,
    # Put the Network
    cyto.Cytoscape(
        id='cytoscape',
        elements=elements,
        style={'width': '100%', 'height': '800px'},
        layout={
            'name': 'preset'
        },
         stylesheet=[
                    {'selector': 'node',
                'style': {'label': 'data(cname)',
                    'background-color': 'data(color)',
                    'shape': 'circle'}},
                     {'selector': 'edge',
                'style': {
                    'line-color': 'data(color)'}}]
    )], style={'flex': 2}),
                    html.Div([# Right part of App - Information
        # Info when clicking a node
        html.P(id='cytoscape-tapNodeData-output'), # General Info
        html.Img(id='bar-graph-matplotlib'), # Bar Plot
        html.P(id='node-presence-output'), # Feature Occurrence
        # Information when clicking an edge
        html.P(id='cytoscape-tapEdgeData-output'),
        # Information on the highlighted network section
        dcc.Markdown('### Information',),
        html.P(id='cytoscape-selected-area-output'),
        # Buttons to save figure
        html.Div('Download graph:'),
        html.Button("as jpg", id="btn-get-jpg"),
        html.Button("as png", id="btn-get-png"),
        html.Button("as svg", id="btn-get-svg")], style={'flex': 1, 'padding': 10})], 
                      
    style={'display': 'flex', 'flexDirection': 'row'}
)


## Functions to respond to changes in the app

@callback(Output('cytoscape', 'stylesheet'),
          Output('cytoscape', 'elements'),
          Output('cytoscape-selected-area-output', 'children'),
          Output(component_id='dropdown-update-layout2', component_property='options'),
              Input('top_mets_chosen', 'value'),
              Input('dropdown-update-layout2', 'value'))
def update_layout(top_chosen, pathway):
    "Updates layout based on the threshold and pathway chosen"

    ### Top chosen metabolites
    top_chosen = int(top_chosen*len(FDiN.nodes()))
    imp_mzs = list(sorted_imps.index[:top_chosen])

    ### Pathway enrichment Analysis as before
    top_ranks_considered = (np.array(list(path_ranks_short.keys())) <= top_chosen).sum()

    # Background Set - Number of Metabolites detected in the FDiN with at least one associated pathway
    total_metabolites_with_pathways = len(path_ranks_short)

    # Get the counts of metabolites of each pathway
    path_assign_hmdb = paths

    # 'Significant' Metabolites - Number of Metabolites in the top ranks considered with at least one associated pathway
    annotated_metabolites_w_path = top_ranks_considered

    # Preparing DF
    over_representation_df = pd.DataFrame(columns=['Pathway Name',
        'Nº of Met. in Dataset', 'Nº of Met. in Pathway', '% of Met. In Set', 'Probability'], dtype='object')

    for pathway2 in tqdm(path_assign_hmdb.index):
        # Pathway Name
        p_name = pathway2
        # Pathway Background Set - Number of Metabolites detected in the FDiN in each pathway
        total_met_in_path = path_assign_hmdb.loc[pathway2]
        # 'Significant' Metabolites in Pathway - Number of Metabolites in the top ranks considered in each pathway
        ann_met_in_path = (np.array(pathway_ranks[pathway2]) <= top_ranks_considered).sum()

        # Calculating probability to find ann_met_in_path or more metabolites in annotated_metabolites_w_path
        prob = stats.hypergeom(M=total_metabolites_with_pathways, 
                                n=total_met_in_path, 
                                N=annotated_metabolites_w_path).sf(ann_met_in_path-1)

        # Adding the line to the DF
        over_representation_df.loc[pathway2] = [p_name, ann_met_in_path, total_met_in_path,
                                               ann_met_in_path/total_met_in_path*100, prob]

    # Sorting DataFrame from least probable to more probable and adding adjusted probability
    # with Benjamini-Hochberg multiple test correction
    over_representation_df_nodes = over_representation_df.sort_values(by='Probability')

    
    #### If no pathway is chosen return
    if pathway == 'None':
        return [], cs_data["elements"]["nodes"] + cs_data["elements"]["edges"], [], [{'label': name.capitalize(), 'value': name}
                                for name in ['None',] + list(over_representation_df_nodes.index)]

    ## If it was chosen, get color, node names to show and opacity of nodes for each
    colors_nodes = {}
    cname_nodes = {}
    opacity_nodes = {}
    for i in FDiN.nodes():
        if i in imp_mzs:
            if pathway in FDiN.nodes()[i]['Pathways']:
                colors_nodes[i] ='red'
            else:
                colors_nodes[i] ='lightcoral'
            cname_nodes[i] = temp_df.loc[i ,'Formula_Assignment']
            opacity_nodes[i] = 1
        else:
            if pathway in FDiN.nodes()[i]['Pathways']:
                colors_nodes[i] ='grey'
                opacity_nodes[i] = 1
            else:
                colors_nodes[i] = 'lightgrey'
                opacity_nodes[i] = 0.4
            cname_nodes[i] =''
    for n, c in zip(cs_data["elements"]["nodes"], colors_nodes.values()):
        n['data']["color"] = c
    for n, c in zip(cs_data["elements"]["nodes"], cname_nodes.values()):
        n['data']["cname"] = c
    for n, c in zip(cs_data["elements"]["nodes"], opacity_nodes.values()):
        n['data']["opacity"] = c

    ## Get color and opacity of edges
    edge_colors = {}
    edge_opacity = {}
    for i in FDiN.edges():
        if i[0] in imp_mzs and i[1] in imp_mzs:
            if pathway in FDiN.edges()[i]['Pathways']:
                edge_colors[i] ='red'
            else:
                edge_colors[i] ='lightcoral'
            edge_opacity[i] = 1
        else:
            if pathway in FDiN.edges()[i]['Pathways']:
                edge_colors[i] ='grey'
                edge_opacity[i] = 1
            else:
                edge_colors[i] = 'lightgrey'
                edge_opacity[i] = 0.4
    for n, c in zip(cs_data["elements"]["edges"], edge_colors.values()):
        n['data']["color"] = c
    for n, c in zip(cs_data["elements"]["edges"], edge_opacity.values()):
        n['data']["opacity"] = c

    # Build the string to describe network component of highlighted area
    construct_string = [html.B(f'Components of top {top_chosen} important metabolites:'), html.Br(), html.Br(),]
    for i in nx.connected_components(FDiN.subgraph(imp_mzs)):
        construct_string.append(html.B(f'{len(i)}-length component'))
        construct_string.append(f'{i}).')
        construct_string.append(html.Br())

    # Return in order to update layout
    return [{'selector': 'node',
                'style': {'label': 'data(cname)',
                    'background-color': 'data(color)',
                    'shape': 'circle',
                    'opacity': 'data(opacity)'}},
            {'selector': 'edge',
                'style': {
                'line-color': 'data(color)',
                'opacity': 'data(opacity)'}}], cs_data["elements"]["nodes"] + cs_data["elements"][
        "edges"], construct_string, [{'label': name.capitalize(), 'value': name}
                                for name in ['None',] + list(over_representation_df_nodes.index)]

@callback(Output('cytoscape', 'layout'),
              Input('dropdown-update-layout', 'value'))
def update_layout(layout):
    return {'name': layout, 'animate': False}

@callback(Output('cytoscape-tapNodeData-output', 'children'),
          Output(component_id='bar-graph-matplotlib', component_property='src'),
          Output(component_id='node-presence-output', component_property='children'),
              Input('cytoscape', 'tapNodeData'))
def displayTapNodeData(data):
    "Display information on the node, plot an average intensity bar chart and display feature occurrence information."
    if data:
        
        ## Node Information Section
        construct_string = [html.B(f'Most recently clicked node: {data["id"]}'), html.Br(), html.Br(),]
        for cat in ['id', 'Formula', 'Name', 'HMDB_ID', 'Names in Pathways',
                    'HMDB_ID in Pathways', 'Pathways', 'SMPDB_IDs']:
            if cat == 'Name':
                # Include VIP score Rank
                construct_string.extend([f'VIP Rank: {str(rank_imp_feats_plsda.loc[data["id"], "VIP"])}',  html.Br(),])
            construct_string.extend([f'{cat.capitalize()}: {str(data[cat])}',  html.Br(),])

        ## Normalized Intensity Bar Plot Section
        plt.close()
        gfinder = processed_data.copy().loc[data["id"], sample_cols]
        percentages = {}
        # Calculate average intensities (and std)
        for cl in classes:
            section = gfinder.iloc[[i for i in range(len(sample_cols)) if target[i]==cl]]#.mean()
            percentages[cl] = section.notnull().sum()/len(section)*100
            gfinder[cl+' Average'] = section.mean()
            gfinder[cl+' sem'] = section.std() / np.sqrt(section.notnull().sum())
        avg_cols = [col for col in gfinder.index if 'Average' in col]
        sem_cols = [col for col in gfinder.index if 'sem' in col]
        # Plot the graph
        bar_plot_info = gfinder.replace({np.nan:0})
        factor = 1
        maxi_v = (np.array(bar_plot_info.loc[avg_cols])*10**factor).max()
        while maxi_v < 1:
            factor+=1
            maxi_v = (np.array(bar_plot_info.loc[avg_cols])*10**factor).max()
        fig, ax = plt.subplots(1,1, figsize=(5,3), constrained_layout=True)
        x = np.arange(len(avg_cols))
        ax.bar(x, np.array(bar_plot_info.loc[avg_cols])*10**factor, color=[colours[0], colours[1], colours[2]],
               yerr=np.array(bar_plot_info.loc[sem_cols])*10**factor, capsize=12)
        ax.set_ylabel(f'Intensity (x$10^{factor}$)', fontsize=15)
        ax.set_title(processed_data.copy().loc[data["id"], 'Formula_Assignment'], fontsize=15)
        ax.set_xticks(x)
        ax.set_xticklabels(classes, fontsize=12)
        # Embed the result in the html output.
        buf = BytesIO()
        fig.savefig(buf, format="png")
        fig_data = base64.b64encode(buf.getbuffer()).decode("ascii")
        fig_bar_matplotlib = f'data:image/png;base64,{fig_data}'

        ## Feature Occurrence Section
        node_pres = [html.B(f'Node Present in % of Class Samples:'), html.Br(), html.Br(),]
        for cl in percentages:
            node_pres.extend([f'{cl}: {percentages[cl]:.3f}', html.Br()])
        
        return construct_string, fig_bar_matplotlib, node_pres
    return '', '', ''

@callback(Output('cytoscape-tapEdgeData-output', 'children'),
              Input('cytoscape', 'tapEdgeData'))
def displayTapEdgeData(data):
    "Show the edge data, that is if it can be mapped to a pathway"
    if data:
        construct_string = [html.B(f'Most recently clicked edge: {data["source"]}-{data["target"]}'),
                            html.Br(), html.Br(),]
        for cat in ['Pathways', 'SMPDB_IDs']:
            construct_string.extend([f'{cat.capitalize()}: {str(data[cat])}',  html.Br(),])
        return construct_string

@callback(
    Output("cytoscape", "generateImage"),
    [Input("btn-get-jpg", "n_clicks"),
        Input("btn-get-png", "n_clicks"),
        Input("btn-get-svg", "n_clicks"),
    ])
def get_image(get_jpg_clicks, get_png_clicks, get_svg_clicks):
    "Download the graph image"
    if ctx.triggered:
        action = "download"
        ftype = ctx.triggered_id.split("-")[-1]

        return {
            'type': ftype,
            'action': action
            }
    else:
        return {
            'type': 'svg',
            'action': 'store'
            }

# Run the app
app.run(debug=True, jupyter_mode='external', port=8058)

# Getting Pathway Graphlets and Single Nodes for Simulations

### *- Important Note: Since the pathway+single nodes sets used for the paper were obtained with another run and even setting the random seed will not get equal graphlets to the ones used in the work, the default is to set this section to False (`obtain_new_graphlets`) so it does not affect posterior steps (it would be necessary to run new models instead of already using the previously trained models).

Finding all graphlets in the Network until size 6 (including)

In [ ]:
# Set True if you want to obtain new graphlets
# Only do it if you aim tio re-run the entire simulations, else do not
obtain_new_graphlets = False

In [ ]:
short_paths = pd.DataFrame(dict(nx.shortest_path_length(FDiN)))
short_paths = short_paths.loc[[i for i in short_paths.columns]]

max_range = 6

node_sets = {1: [[n,] for n in FDiN.nodes()]}
short_paths = pd.DataFrame(dict(nx.shortest_path_length(FDiN)))

for r in range(2, max_range+1):
    node_sets[r] = []
    temp_g = FDiN.copy()
    all_past_paths = node_sets[r-1].copy()
    short_paths_copy = short_paths.copy()
    for node in tqdm(FDiN.nodes()):
        new_paths = []
        paths_to_remove = []
        past_paths = []
        for i in all_past_paths:
            if node in i:
                past_paths.append(i)
                paths_to_remove.append(i)
        for p in paths_to_remove:
            all_past_paths.remove(p)
        idxs = short_paths_copy.loc[short_paths_copy[node] <= r].index
        temp_g_sg = temp_g.subgraph(idxs)
        for path in past_paths:
            for c_n in path:
                for n in temp_g_sg.neighbors(c_n):
                    if n not in path:
                        new_path = path + [n, ]
                        new_path.sort()
                        #if new_path not in new_paths:
                        new_paths.append(new_path)
        for i in range(len(new_paths)):
            new_paths[i] = tuple(new_paths[i])
        new_paths = pd.unique(pd.Series(new_paths))
        new_paths = [list(i) for i in new_paths]
        node_sets[r].extend(new_paths)
        temp_g.remove_node(node)
        short_paths_copy = short_paths_copy.drop(node)

In [ ]:
for i in node_sets:
    print(f'{i}-length graphlets: {len(node_sets[i])} paths.')

## 5-Length Graphlets

1) Obtain all sets of pathway graphlets.

2) Obtain all sets of single nodes.

3) Obtain the node to be the gap in each pathway graphlet.

4) Save all sets.

In [ ]:
# Getting the graphlet sets
np.random.seed(45840)
# Number of sets to obtain
n_sets = 16

if obtain_new_graphlets:
    # Get the possible paths
    sets_possible = node_sets[5].copy()
    graphlets = []
    print(len(sets_possible))
    a=0
    # Until we get the number 
    while a < n_sets:

        # Choose a possible graphlet within the set
        choice = np.random.choice(range(len(sets_possible)))
        set_chosen = sets_possible[choice]
        graphlets.append(set_chosen)

        # Remove all graphlets from the list that have at least a node from the graphlet
        new_list = []
        for i in sets_possible:
            if len(np.intersect1d(set_chosen, i)) == 0:
                new_list.append(i)
        sets_possible = new_list
        print(len(sets_possible))
        a+=1

    for i in graphlets:
        print(i)

In [ ]:
# Getting the single node sets
np.random.seed(40012)

if obtain_new_graphlets:
    # Set up base
    sample_dict = dict(zip(sample_cols, target))
    cp_samples = [i for i in sample_dict.keys() if sample_dict[i] == 'Cold-phase']
    rep_samples = [i for i in sample_dict.keys() if sample_dict[i] == 'Reperfusion']
    all_single_nodes = []

    a = 0

    # For each graphlet
    for graphlet in graphlets:

        all_nodes = list(FDiN.nodes())
        nodes_to_remove = []
        nodes_to_select = all_nodes.copy()
        # Remove graphlet noeds and their neighbors
        for i in graphlet:
            for n in FDiN.neighbors(i):
                if n not in nodes_to_remove:
                    nodes_to_remove.append(n)
                    nodes_to_select.remove(n)
        
        # Then add a single node. For each node added, remove its neighbors from the possibilities
        single_nodes = []
        for i in range(5):
            chosen_node = np.random.choice(nodes_to_select)
            single_nodes.append(chosen_node)
            for n in FDiN.neighbors(chosen_node):
                if n not in nodes_to_remove:
                    nodes_to_remove.append(n)
                    nodes_to_select.remove(n)
        all_single_nodes.append(single_nodes)

        a += 1

In [ ]:
# Getting the gaps to use in each of the graphlet sets
np.random.seed(594133)

special_nodes = []
if obtain_new_graphlets:
    # For each graphlet
    for i in range(len(graphlets)):
        gr = graphlets[i]
        # Subgraph the graphlet and see the nodes with the highest degree, select one of them
        g_graph = FDiN.subgraph(gr)
        degs = pd.Series(dict(g_graph.degree()))
        degs = degs[degs>= degs.max()]
        special_nodes.append(np.random.choice(degs.index))
special_nodes

In [ ]:
# Save all sets obtained
if obtain_new_graphlets:
    with open('MD_Simulations/Data/MD_GNNPathwayTest_graphlets_size5.txt', 'w') as a:
        for i in graphlets:
            a.write(', '.join(i))
            a.write('\n')

    with open('MD_Simulations/Data/MD_GNNPathwayTest_singlenodes_size5.txt', 'w') as a:
        for i in all_single_nodes:
            a.write(', '.join(i))
            a.write('\n')

    with open('MD_Simulations/Data/MD_5Graphlets_gaps.txt', 'w') as a:
        for i in special_nodes:
            a.write(i)
            a.write('\n')

## 6-Length Graphlets

1) Obtain all sets of pathway graphlets.

2) Obtain all sets of single nodes.

3) Obtain the node to be the gap in each pathway graphlet.

4) Save all sets.

In [ ]:
# Getting the graphlet sets
np.random.seed(45840)
# Number of sets to obtain
n_sets = 16

if obtain_new_graphlets:
    # Get the possible paths
    sets_possible = node_sets[6].copy()
    graphlets = []
    print(len(sets_possible))
    a=0
    # Until we get the number 
    while a < n_sets:

        # Choose a possible graphlet within the set
        choice = np.random.choice(range(len(sets_possible)))
        set_chosen = sets_possible[choice]
        graphlets.append(set_chosen)

        # Remove all graphlets from the list that have at least a node from the graphlet
        new_list = []
        for i in sets_possible:
            if len(np.intersect1d(set_chosen, i)) == 0:
                new_list.append(i)
        sets_possible = new_list
        print(len(sets_possible))
        a+=1

    for i in graphlets:
        print(i)

In [ ]:
# Getting the single node sets
np.random.seed(40012)

if obtain_new_graphlets:
    # Set up base
    all_single_nodes = []

    a = 0

    # For each graphlet
    for graphlet in graphlets:

        all_nodes = list(FDiN.nodes())
        nodes_to_remove = []
        nodes_to_select = all_nodes.copy()
        # Remove graphlet noeds and their neighbors
        for i in graphlet:
            for n in FDiN.neighbors(i):
                if n not in nodes_to_remove:
                    nodes_to_remove.append(n)
                    nodes_to_select.remove(n)
        
        # Then add a single node. For each node added, remove its neighbors from the possibilities
        single_nodes = []
        for i in range(5):
            chosen_node = np.random.choice(nodes_to_select)
            single_nodes.append(chosen_node)
            for n in FDiN.neighbors(chosen_node):
                if n not in nodes_to_remove:
                    nodes_to_remove.append(n)
                    nodes_to_select.remove(n)
        all_single_nodes.append(single_nodes)

        a += 1

In [ ]:
# Getting the gaps to use in each of the graphlet sets
np.random.seed(594133)

special_nodes = []
if obtain_new_graphlets:
    # For each graphlet
    for i in range(len(graphlets)):
        gr = graphlets[i]
        # Subgraph the graphlet and see the nodes with the highest degree, select one of them
        g_graph = FDiN.subgraph(gr)
        degs = pd.Series(dict(g_graph.degree()))
        degs = degs[degs>= degs.max()]
        special_nodes.append(np.random.choice(degs.index))
special_nodes

In [ ]:
# Save all sets obtained
if obtain_new_graphlets:
    with open('MD_Simulations/Data/MD_GNNPathwayTest_graphlets_size6.txt', 'w') as a:
        for i in graphlets:
            a.write(', '.join(i))
            a.write('\n')

    with open('MD_Simulations/Data/MD_GNNPathwayTest_singlenodes_size6.txt', 'w') as a:
        for i in all_single_nodes:
            a.write(', '.join(i))
            a.write('\n')

    with open('MD_Simulations/Data/MD_6Graphlets_gaps.txt', 'w') as a:
        for i in special_nodes:
            a.write(i)
            a.write('\n')

## 4-Length Graphlets

1) Obtain all sets of pathway graphlets.

2) Obtain all sets of single nodes.

3) Save all sets.

In [ ]:
# Getting the graphlet sets
np.random.seed(45840)
# Number of sets to obtain
n_sets = 16

if obtain_new_graphlets:
    # Get the possible paths
    sets_possible = node_sets[4].copy()
    graphlets = []
    print(len(sets_possible))
    a=0
    # Until we get the number 
    while a < n_sets:

        # Choose a possible graphlet within the set
        choice = np.random.choice(range(len(sets_possible)))
        set_chosen = sets_possible[choice]
        graphlets.append(set_chosen)

        # Remove all graphlets from the list that have at least a node from the graphlet
        new_list = []
        for i in sets_possible:
            if len(np.intersect1d(set_chosen, i)) == 0:
                new_list.append(i)
        sets_possible = new_list
        print(len(sets_possible))
        a+=1

    for i in graphlets:
        print(i)

In [ ]:
# Getting the single node sets
np.random.seed(40012)

if obtain_new_graphlets:
    # Set up base
    all_single_nodes = []

    a = 0

    # For each graphlet
    for graphlet in graphlets:

        all_nodes = list(FDiN.nodes())
        nodes_to_remove = []
        nodes_to_select = all_nodes.copy()
        # Remove graphlet noeds and their neighbors
        for i in graphlet:
            for n in FDiN.neighbors(i):
                if n not in nodes_to_remove:
                    nodes_to_remove.append(n)
                    nodes_to_select.remove(n)
        
        # Then add a single node. For each node added, remove its neighbors from the possibilities
        single_nodes = []
        for i in range(5):
            chosen_node = np.random.choice(nodes_to_select)
            single_nodes.append(chosen_node)
            for n in FDiN.neighbors(chosen_node):
                if n not in nodes_to_remove:
                    nodes_to_remove.append(n)
                    nodes_to_select.remove(n)
        all_single_nodes.append(single_nodes)

        a += 1

In [ ]:
# Save all sets obtained
if obtain_new_graphlets:
    with open('MD_Simulations/Data/MD_GNNPathwayTest_graphlets_size4.txt', 'w') as a:
        for i in graphlets:
            a.write(', '.join(i))
            a.write('\n')

    with open('MD_Simulations/Data/MD_GNNPathwayTest_singlenodes_size4.txt', 'w') as a:
        for i in all_single_nodes:
            a.write(', '.join(i))
            a.write('\n')